# Retrieval, Hybrid Search, and Reranking in Modern AI

notebook demonstrates:
- Standard vector retrieval (cosine similarity, FAISS)
- Hybrid search (semantic + keyword)
- Reranking approaches for improved relevance

## Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install langchain faiss-cpu sentence-transformers tiktoken
# !pip install rank-bm25 (TF-IDF vector)

In [1]:
# Imports
from langchain_classic.vectorstores import FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document
from rank_bm25 import BM25Okapi
import numpy as np
import re

## Sample Documents

In [2]:
# Create sample documents
sample_docs = [
    Document(page_content="Python is a popular programming language for data science and AI."),
    Document(page_content="FAISS enables efficient similarity search in high-dimensional spaces."),
    Document(page_content="BM25 is a ranking function used by search engines to estimate relevance."),
    Document(page_content="Hybrid search combines semantic and keyword-based retrieval for better results."),
    Document(page_content="Cosine similarity measures the angle between two vectors in vector space."),
    Document(page_content="Reranking can improve search results by applying a second scoring step.")
]

## Standard Vector Retrieval (Cosine, FAISS)

In [3]:
# Embedding model (using a small, fast model for demo)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\reach\AppData\Local\Temp\ipykernel_27272\331036842.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [4]:
# Split documents if needed (here, each doc is short)
# For real use, use text_splitter.split_documents(...)

# Create FAISS vector store
vector_store = FAISS.from_documents(sample_docs, embeddings)

In [5]:
# Simple retrieval function

def vector_search(query, k=3):
    
    print(f"\n🔍 Query: {query}")
    
    results = vector_store.similarity_search_with_score(query, k=k)
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"{i}. {doc.page_content} (Score: {score:.4f})")
        
    return results

# Test vector search
vector_search("What is cosine similarity?")
vector_search("How to combine semantic and keyword search?")


🔍 Query: What is cosine similarity?
1. Cosine similarity measures the angle between two vectors in vector space. (Score: 0.3487)
2. FAISS enables efficient similarity search in high-dimensional spaces. (Score: 1.3044)
3. Hybrid search combines semantic and keyword-based retrieval for better results. (Score: 1.6755)

🔍 Query: How to combine semantic and keyword search?
1. Hybrid search combines semantic and keyword-based retrieval for better results. (Score: 0.5437)
2. Reranking can improve search results by applying a second scoring step. (Score: 1.3101)
3. BM25 is a ranking function used by search engines to estimate relevance. (Score: 1.3246)


[(Document(id='19acb04c-20c0-4c47-ac51-9f6b3864e1c7', metadata={}, page_content='Hybrid search combines semantic and keyword-based retrieval for better results.'),
  np.float32(0.5436871)),
 (Document(id='45c9b1de-611f-43d2-a300-5d98398153e1', metadata={}, page_content='Reranking can improve search results by applying a second scoring step.'),
  np.float32(1.3101056)),
 (Document(id='dfebb4d7-d5e0-440b-a05c-747c7212ec89', metadata={}, page_content='BM25 is a ranking function used by search engines to estimate relevance.'),
  np.float32(1.3245773))]

## Hybrid Search (Semantic + Keyword)

In [6]:
# Prepare BM25 (keyword) index
corpus           = [doc.page_content for doc in sample_docs]
tokenized_corpus = [re.findall(r"\w+", doc.lower()) for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(query, k=3, alpha=0.5):
    """
    alpha: weight for semantic (vector) score (0-1)
    (1-alpha): weight for keyword (BM25) score
    """
    # Semantic scores
    vector_results = vector_store.similarity_search_with_score(query, k=len(sample_docs))
    vector_scores  = {doc.page_content: 1-score for doc, score in vector_results}  # Higher is better
    
    # BM25 scores
    tokenized_query = re.findall(r"\w+", query.lower())
    bm25_scores     = bm25.get_scores(tokenized_query)
    bm25_dict       = {corpus[i]: bm25_scores[i] for i in range(len(corpus))}
    
    # Combine
    combined = []
    
    for doc in corpus:
        score = alpha * vector_scores.get(doc, 0) + (1-alpha) * bm25_dict.get(doc, 0)
        combined.append((doc, score))
        
    combined.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n Hybrid Search for: {query}")
    for i, (doc, score) in enumerate(combined[:k], 1):
        print(f"{i}. {doc} (Hybrid Score: {score:.4f})")
    return combined[:k]

# Test hybrid search
hybrid_search("cosine similarity", alpha=0.5)
hybrid_search("semantic keyword search", alpha=0.7)


 Hybrid Search for: cosine similarity
1. Cosine similarity measures the angle between two vectors in vector space. (Hybrid Score: 1.2687)
2. FAISS enables efficient similarity search in high-dimensional spaces. (Hybrid Score: 0.1664)
3. Hybrid search combines semantic and keyword-based retrieval for better results. (Hybrid Score: -0.3638)

 Hybrid Search for: semantic keyword search
1. Hybrid search combines semantic and keyword-based retrieval for better results. (Hybrid Score: 1.1451)
2. BM25 is a ranking function used by search engines to estimate relevance. (Hybrid Score: -0.0108)
3. FAISS enables efficient similarity search in high-dimensional spaces. (Hybrid Score: -0.0899)


[('Hybrid search combines semantic and keyword-based retrieval for better results.',
  np.float64(1.1451183714310844)),
 ('BM25 is a ranking function used by search engines to estimate relevance.',
  np.float64(-0.0108009609934)),
 ('FAISS enables efficient similarity search in high-dimensional spaces.',
  np.float64(-0.0898512349438118))]

## Reranking Approaches

In [7]:
# Simple reranking: rerank top-N hybrid results by semantic score only

def rerank_by_semantic(query, hybrid_results, top_n=3):
    print(f"\n Reranking top {len(hybrid_results)} by semantic score:")
    
    # Get semantic scores for these docs
    docs           = [doc for doc, _ in hybrid_results]
    vector_results = vector_store.similarity_search_with_score(query, k=len(docs))
    
    # Only keep docs in rerank set
    rerank_scores = {doc.page_content: 1-score for doc, score in vector_results if doc.page_content in docs}
    reranked      = sorted(hybrid_results, key=lambda x: rerank_scores.get(x[0], 0), reverse=True)
    
    for i, (doc, score) in enumerate(reranked[:top_n], 1):
        print(f"{i}. {doc} (Semantic Score: {rerank_scores.get(doc, 0):.4f})")
        
    return reranked[:top_n]

# Example: rerank hybrid results
hybrid_results = hybrid_search("cosine similarity", alpha=0.5)
rerank_by_semantic("cosine similarity", hybrid_results, top_n=3)


 Hybrid Search for: cosine similarity
1. Cosine similarity measures the angle between two vectors in vector space. (Hybrid Score: 1.2687)
2. FAISS enables efficient similarity search in high-dimensional spaces. (Hybrid Score: 0.1664)
3. Hybrid search combines semantic and keyword-based retrieval for better results. (Hybrid Score: -0.3638)

 Reranking top 3 by semantic score:
1. Cosine similarity measures the angle between two vectors in vector space. (Semantic Score: 0.6634)
2. FAISS enables efficient similarity search in high-dimensional spaces. (Semantic Score: -0.3035)
3. Hybrid search combines semantic and keyword-based retrieval for better results. (Semantic Score: -0.7276)


[('Cosine similarity measures the angle between two vectors in vector space.',
  np.float64(1.2687488805828466)),
 ('FAISS enables efficient similarity search in high-dimensional spaces.',
  np.float64(0.16636949471582707)),
 ('Hybrid search combines semantic and keyword-based retrieval for better results.',
  np.float64(-0.36378705501556396))]

## Summary
- Standard vector retrieval (FAISS, cosine) is fast and effective for semantic similarity
- Hybrid search combines strengths of semantic and keyword search
- Reranking can further improve result relevance